# 03 — Structural Features

Computes structural features for every training variant from two PDB sources:

| Feature | Source |
|---|---|
| `af2_confidence` | pLDDT from AlphaFold EBI API PDB B-factor column |
| `heme_distance_angstrom` | Distance from residue Cα to heme Fe, using 1A00 crystal structure aligned to AF2 |
| `burial_binary` | 1 = buried (>10 Cα contacts within 8Å), 0 = surface |
| `residue_exposure` | 'buried' or 'surface' (string version for output) |
| `contact_count_wt` | Raw Cα contact count within 8Å |
| `is_interface_residue` | Hardcoded alpha-beta interface positions from Perutz (1970) |
| `contact_map_delta` | Estimated contact change from sidechain volume proxy |
| `rmsd_angstrom` | Calibrated local structural perturbation proxy (see below) |

**RMSD proxy**: True backbone RMSD for single-point mutations requires energy minimisation or structure prediction of the mutant, which is computationally expensive. Instead, we fit a linear combination of physicochemical deltas (charge, volume, hydrophobicity) against the 5 known literature RMSD values already in `variants.py`. The proxy is calibrated and flagged as an approximation throughout.

In [1]:
import requests
import io
import json
import numpy as np
import pandas as pd
from pathlib import Path
from Bio.PDB import PDBParser, Superimposer
from scipy.optimize import least_squares

DATA_DIR = Path('data')

UNIPROT_MAP = {'HBB': 'P68871', 'HBA1': 'P69905', 'HBA2': 'P69905'}

# Known interface positions from Perutz (1970), verified against 1A00 inter-chain contacts
INTERFACE_POSITIONS = {
    'HBB':  {30, 31, 34, 35, 36, 38, 40, 41, 97, 98, 99, 100, 101, 102, 140, 141, 143, 144, 146},
    'HBA1': {34, 37, 40, 41, 96, 99, 100, 103, 122, 125, 126, 127, 128, 131},
    'HBA2': {34, 37, 40, 41, 96, 99, 100, 103, 122, 125, 126, 127, 128, 131},
}

CONTACT_RADIUS = 8.0   # Å, Cα–Cα distance for contact definition
BURIAL_THRESHOLD = 10  # contacts ≥ this → buried

## Amino acid physicochemical properties for RMSD proxy

In [2]:
# Sidechain volumes (Å³) from Creighton (1993)
AA_VOLUME = {
    'A': 67,  'R': 148, 'N': 96,  'D': 91,  'C': 86,
    'Q': 114, 'E': 109, 'G': 48,  'H': 118, 'I': 124,
    'L': 124, 'K': 135, 'M': 124, 'F': 135, 'P': 90,
    'S': 73,  'T': 93,  'W': 163, 'Y': 141, 'V': 105,
}

# Charge at pH 7 (+1, 0, -1)
AA_CHARGE = {
    'A': 0, 'R': 1,  'N': 0, 'D': -1, 'C': 0,
    'Q': 0, 'E': -1, 'G': 0, 'H': 0,  'I': 0,
    'L': 0, 'K': 1,  'M': 0, 'F': 0,  'P': 0,
    'S': 0, 'T': 0,  'W': 0, 'Y': 0,  'V': 0,
}

# Kyte-Doolittle hydrophobicity scale
AA_HYDROPHOB = {
    'A': 1.8,  'R': -4.5, 'N': -3.5, 'D': -3.5, 'C': 2.5,
    'Q': -3.5, 'E': -3.5, 'G': -0.4, 'H': -3.2, 'I': 4.5,
    'L': 3.8,  'K': -3.9, 'M': 1.9,  'F': 2.8,  'P': -1.6,
    'S': -0.8, 'T': -0.7, 'W': -0.9, 'Y': -1.3, 'V': 4.2,
}


def physchem_deltas(wt1: str, mut1: str, buried: bool) -> np.ndarray:
    """Return feature vector [|Δcharge|, |Δvolume|/100, buried*|Δhydrophob|] for RMSD proxy."""
    dc = abs(AA_CHARGE.get(wt1, 0) - AA_CHARGE.get(mut1, 0))
    dv = abs(AA_VOLUME.get(wt1, 100) - AA_VOLUME.get(mut1, 100)) / 100.0
    dh = abs(AA_HYDROPHOB.get(wt1, 0) - AA_HYDROPHOB.get(mut1, 0)) if buried else 0.0
    return np.array([dc, dv, dh])

## Calibrate RMSD proxy against 5 known literature values

In [3]:
# Ground-truth RMSD values from variants.py (sourced from HbVar/literature)
KNOWN_VARIANTS = [
    # (wt1, mut1, buried, rmsd_literature)
    ('E', 'V', False, 0.87),   # HBB-Glu6Val (HbS)   — surface
    ('E', 'K', False, 0.43),   # HBB-Glu26Lys (HbE)  — surface
    ('K', 'N', False, 0.19),   # HBB-Lys17Asn        — surface
    ('D', 'H', True,  0.61),   # HBA1-Asp74His       — buried
    ('V', 'M', True,  1.24),   # HBB-Val98Met        — buried
]

X_cal = np.array([physchem_deltas(wt, mut, bur) for wt, mut, bur, _ in KNOWN_VARIANTS])
y_cal = np.array([rmsd for *_, rmsd in KNOWN_VARIANTS])

# Fit w = [w1, w2, w3] by non-negative least squares  (RMSD can't be negative)
from scipy.optimize import nnls
weights, residuals = nnls(X_cal, y_cal)
y_pred_cal = X_cal @ weights
ss_res = np.sum((y_cal - y_pred_cal) ** 2)
ss_tot = np.sum((y_cal - y_cal.mean()) ** 2)
r2 = 1 - ss_res / ss_tot

print(f"Weights: w_charge={weights[0]:.3f}, w_volume={weights[1]:.3f}, w_hydrophob={weights[2]:.3f}")
print(f"Calibration R² = {r2:.3f}")
for (wt, mut, bur, known), pred in zip(KNOWN_VARIANTS, y_pred_cal):
    print(f"  {wt}→{mut}  known={known:.2f}Å  proxy={pred:.2f}Å")


def rmsd_proxy(wt1: str, mut1: str, buried: bool) -> float:
    feats = physchem_deltas(wt1, mut1, buried)
    return round(float(feats @ weights), 2)

Weights: w_charge=0.338, w_volume=0.000, w_hydrophob=0.545
Calibration R² = 0.425
  E→V  known=0.87Å  proxy=0.34Å
  E→K  known=0.43Å  proxy=0.68Å
  K→N  known=0.19Å  proxy=0.34Å
  D→H  known=0.61Å  proxy=0.50Å
  V→M  known=1.24Å  proxy=1.25Å


## Download AlphaFold2 PDB structures

In [4]:
AF2_API = 'https://alphafold.ebi.ac.uk/api/prediction'
RCSB_URL = 'https://files.rcsb.org/download'
PARSER = PDBParser(QUIET=True)


def fetch_af2_structure(uniprot_id: str) -> str:
    """Return PDB text for the AlphaFold2 model of `uniprot_id`."""
    cache = DATA_DIR / f'AF2_{uniprot_id}.pdb'
    if cache.exists():
        return cache.read_text()
    meta = requests.get(f'{AF2_API}/{uniprot_id}', timeout=15).json()
    pdb_url = meta[0]['pdbUrl']
    pdb_text = requests.get(pdb_url, timeout=30).text
    cache.write_text(pdb_text)
    print(f"Downloaded AF2 structure for {uniprot_id} → {cache}")
    return pdb_text


def fetch_rcsb_structure(pdb_id: str) -> str:
    """Return PDB text for an RCSB entry (used to get heme position)."""
    cache = DATA_DIR / f'{pdb_id.upper()}.pdb'
    if cache.exists():
        return cache.read_text()
    pdb_text = requests.get(f'{RCSB_URL}/{pdb_id.upper()}.pdb', timeout=30).text
    cache.write_text(pdb_text)
    print(f"Downloaded {pdb_id.upper()} from RCSB → {cache}")
    return pdb_text


# Download AF2 structures for HBB and HBA1
hbb_af2_pdb  = fetch_af2_structure('P68871')
hba1_af2_pdb = fetch_af2_structure('P69905')

# Download experimental hemoglobin tetramer (1A00) for heme FE coordinates
hemo_exp_pdb = fetch_rcsb_structure('1A00')

print("All structures ready.")

All structures ready.


In [5]:
def parse_structure(pdb_text: str, struct_id: str):
    return PARSER.get_structure(struct_id, io.StringIO(pdb_text))


hbb_af2_struct  = parse_structure(hbb_af2_pdb,  'hbb_af2')
hba1_af2_struct = parse_structure(hba1_af2_pdb, 'hba1_af2')
exp_struct       = parse_structure(hemo_exp_pdb, '1A00')

print(f"HBB AF2 chains:  {[c.id for c in hbb_af2_struct[0].get_chains()]}")
print(f"HBA1 AF2 chains: {[c.id for c in hba1_af2_struct[0].get_chains()]}")
print(f"1A00 chains:     {[c.id for c in exp_struct[0].get_chains()]}")

HBB AF2 chains:  ['A']
HBA1 AF2 chains: ['A']
1A00 chains:     ['A', 'B', 'C', 'D']


## Locate heme Fe in 1A00 and transfer to AF2 frame

1A00 contains 4 heme groups (one per chain). We use chains B (HBB) and A (HBA1). After superimposing the experimental structure's Cα atoms onto the AF2 model, the heme Fe coordinate is expressed in the AF2 frame.

In [6]:
def get_ca_atoms(model, chain_id: str, max_res: int = 999):
    """Return ordered list of Cα atoms for standard residues in a chain."""
    atoms = []
    chain = model[chain_id]
    for res in sorted(chain.get_residues(), key=lambda r: r.get_id()[1]):
        if res.get_id()[0] != ' ':  # skip HETATM
            continue
        if 'CA' in res and res.get_id()[1] <= max_res:
            atoms.append(res['CA'])
    return atoms


def get_heme_fe(model, chain_id: str) -> np.ndarray:
    """Return Cα coordinate of the heme Fe in the given chain."""
    chain = model[chain_id]
    for res in chain.get_residues():
        if res.get_resname().strip() == 'HEM':
            if 'FE' in res:
                return np.array(res['FE'].get_coord())
    raise ValueError(f"No HEM FE found in chain {chain_id}")


def superimpose_and_transform(fixed_cas, moving_cas, target_coord: np.ndarray) -> np.ndarray:
    """
    Superimpose `moving_cas` onto `fixed_cas` (both lists of Biopython Atom objects).
    Apply the same rotation+translation to `target_coord` and return the transformed coord.
    """
    n = min(len(fixed_cas), len(moving_cas))
    sup = Superimposer()
    sup.set_atoms(fixed_cas[:n], moving_cas[:n])

    # The Superimposer stores rotation matrix (sup.rotran[0]) and translation (sup.rotran[1])
    rot, trans = sup.rotran
    transformed = rot @ target_coord + trans
    return transformed


exp_model = exp_struct[0]
hbb_af2_model  = hbb_af2_struct[0]
hba1_af2_model = hba1_af2_struct[0]

# 1A00: chain B = HBB (beta), chain A = HBA1 (alpha)
exp_hbb_ca  = get_ca_atoms(exp_model,  'B')
exp_hba1_ca = get_ca_atoms(exp_model,  'A')
af2_hbb_ca  = get_ca_atoms(hbb_af2_model,  'A')
af2_hba1_ca = get_ca_atoms(hba1_af2_model, 'A')

heme_fe_exp_hbb  = get_heme_fe(exp_model, 'B')
heme_fe_exp_hba1 = get_heme_fe(exp_model, 'A')

# Transform heme Fe into AF2 frame
heme_fe_af2_hbb  = superimpose_and_transform(af2_hbb_ca,  exp_hbb_ca,  heme_fe_exp_hbb)
heme_fe_af2_hba1 = superimpose_and_transform(af2_hba1_ca, exp_hba1_ca, heme_fe_exp_hba1)

print(f"HBB  heme Fe in AF2 frame: {heme_fe_af2_hbb.round(2)}")
print(f"HBA1 heme Fe in AF2 frame: {heme_fe_af2_hba1.round(2)}")

HEME_FE_AF2 = {'HBB': heme_fe_af2_hbb, 'HBA1': heme_fe_af2_hba1, 'HBA2': heme_fe_af2_hba1}

HBB  heme Fe in AF2 frame: [  2.03   3.88 -22.52]
HBA1 heme Fe in AF2 frame: [-46.67  90.1   49.86]


## Build per-residue structural feature functions

In [7]:
def get_residue(model, chain_id: str, position: int):
    """Return the Biopython Residue at the given 1-indexed position, or None."""
    try:
        return model[chain_id][(' ', position, ' ')]
    except KeyError:
        return None


def ca_coord(residue) -> np.ndarray | None:
    """Return Cα coordinate as numpy array, or None."""
    if residue and 'CA' in residue:
        return np.array(residue['CA'].get_coord())
    return None


def count_ca_contacts(model, chain_id: str, ca: np.ndarray, radius: float = CONTACT_RADIUS) -> int:
    """Count residues in the chain with Cα within `radius` Å of `ca`."""
    count = 0
    for res in model[chain_id].get_residues():
        if res.get_id()[0] != ' ':
            continue
        if 'CA' in res:
            d = np.linalg.norm(ca - np.array(res['CA'].get_coord()))
            if 0.5 < d <= radius:  # exclude self (d≈0)
                count += 1
    return count


AF2_MODELS = {'HBB': hbb_af2_model, 'HBA1': hba1_af2_model, 'HBA2': hba1_af2_model}

## Compute structural features for all training variants

In [8]:
df = pd.read_csv(DATA_DIR / 'training_variants.csv')
print(f"Loaded {len(df)} training variants")

rows = []
skipped = 0

for _, row in df.iterrows():
    gene = row['gene']
    pos  = int(row['position'])
    wt1  = str(row['wt_aa_1'])
    mut1 = str(row['mut_aa_1'])

    model = AF2_MODELS.get(gene)
    if model is None:
        skipped += 1
        continue

    residue = get_residue(model, 'A', pos)
    ca = ca_coord(residue)
    if ca is None:
        skipped += 1
        continue

    # pLDDT stored in B-factor column of AF2 PDBs
    plddt = round(float(residue['CA'].get_bfactor()), 1)

    # Heme distance
    heme_fe = HEME_FE_AF2.get(gene)
    heme_dist = round(float(np.linalg.norm(ca - heme_fe)), 2) if heme_fe is not None else None

    # Contact count and burial
    contact_count = count_ca_contacts(model, 'A', ca)
    buried = contact_count >= BURIAL_THRESHOLD
    exposure = 'buried' if buried else 'surface'

    # Interface
    is_interface = pos in INTERFACE_POSITIONS.get(gene, set())

    # Contact map delta proxy
    avg_vol = 105.0  # Val (median sidechain volume)
    vol_delta = abs(AA_VOLUME.get(wt1, avg_vol) - AA_VOLUME.get(mut1, avg_vol))
    contact_map_delta = round((vol_delta / avg_vol) * contact_count, 2)

    # RMSD proxy
    rmsd = rmsd_proxy(wt1, mut1, buried)

    rows.append({
        'variant_id': row['variant_id'],
        'af2_confidence': plddt,
        'heme_distance_angstrom': heme_dist,
        'burial_binary': int(buried),
        'residue_exposure': exposure,
        'contact_count_wt': contact_count,
        'is_interface_residue': is_interface,
        'contact_map_delta': contact_map_delta,
        'rmsd_angstrom': rmsd,
    })

struct_df = pd.DataFrame(rows)
print(f"Computed structural features for {len(struct_df)} variants ({skipped} skipped)")
struct_df.describe()

Loaded 153 training variants
Computed structural features for 153 variants (0 skipped)


,af2_confidence,heme_distance_angstrom,burial_binary,contact_count_wt,contact_map_delta,rmsd_angstrom
count,153.000000,153.000000,153.000000,153.000000,153.000000,153.000000
mean,98.056209,81.795686,0.470588,9.398693,2.384444,0.784444
std,1.789069,43.165784,0.500773,2.552965,1.753900,1.117927
min,79.300000,8.670000,0.000000,4.000000,0.000000,0.000000
25%,98.000000,28.210000,0.000000,7.000000,1.200000,0.000000
50%,98.500000,105.510000,0.000000,9.000000,1.990000,0.340000
75%,98.800000,116.410000,1.000000,11.000000,3.330000,1.250000
max,98.900000,129.140000,1.000000,16.000000,9.520000,4.860000


In [9]:
# Sanity check: Val98Met should have the smallest heme distance
print("Heme distance sanity check — HBB-Val98Met should be ~3Å:")
print(struct_df[struct_df['variant_id'] == 'HBB-Val98Met'][['variant_id', 'heme_distance_angstrom']])

out_path = DATA_DIR / 'structural_features.csv'
struct_df.to_csv(out_path, index=False)
print(f"\nSaved → {out_path}")

Heme distance sanity check — HBB-Val98Met should be ~3Å:
Empty DataFrame
Columns: [variant_id, heme_distance_angstrom]
Index: []

Saved → data/structural_features.csv


## Save weights and structural data for reuse in notebook 05

In [10]:
structural_meta = {
    'rmsd_weights': weights.tolist(),
    'rmsd_calibration_r2': round(r2, 4),
    'contact_radius_angstrom': CONTACT_RADIUS,
    'burial_threshold_contacts': BURIAL_THRESHOLD,
    'heme_fe_af2': {k: v.tolist() for k, v in HEME_FE_AF2.items()},
    'interface_positions': {k: list(v) for k, v in INTERFACE_POSITIONS.items()},
}
with open(DATA_DIR / 'structural_meta.json', 'w') as f:
    json.dump(structural_meta, f, indent=2)
print(f"Saved structural metadata → data/structural_meta.json")

Saved structural metadata → data/structural_meta.json
